# 第7讲：分类算法与工程风险识别

## Goal

在计划开工前14天，根据当时可得信息筛查延误超过7天的工作包，并理解阈值怎样改变漏报、误报和复核量。

## Setup

- 一行：一个工作包在分析截点的快照；
- 正类：延误超过7天；
- 分组：按`project_id`划分；
- 模型选择：验证AP；
- 阈值选择：只使用验证集。

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

plt.rcParams['figure.figsize']=(9,4.5)
plt.rcParams['axes.grid']=True
plt.rcParams['grid.alpha']=0.22


## 1. 参数与数据

In [ ]:
THRESHOLD = 0.50  # 学生先运行0.50，再改为0.28
SAVE_FIGURES = False
DATA_PATH=Path('data/DS-C07_工作包延误风险.csv')
SPLIT_PATH=Path('data/DS-C07_项目冻结划分.csv')
FIGURE_DIR=Path('figures')
if SAVE_FIGURES: FIGURE_DIR.mkdir(exist_ok=True)


## 2. 读取数据并核对项目划分

In [ ]:
raw=pd.read_csv(DATA_PATH)
split_map=pd.read_csv(SPLIT_PATH)
data=raw.merge(split_map,on='project_id',how='left',validate='many_to_one')
summary=data.groupby('split').agg(rows=('work_package_id','size'),projects=('project_id','nunique'),positives=('target_delay_over_7d','sum'),positive_rate=('target_delay_over_7d','mean')).reindex(['train','validation','test'])
display(summary.round(3))
assert raw.shape==(2400,14)
assert raw['project_id'].nunique()==200
sets={n:set(g['project_id']) for n,g in data.groupby('split')}
assert sets['train'].isdisjoint(sets['validation']) and sets['train'].isdisjoint(sets['test']) and sets['validation'].isdisjoint(sets['test'])


## 3. 定义特征、模型和指标

In [ ]:
TARGET='target_delay_over_7d'
CATEGORICAL=['discipline']
NUMERIC=['planned_duration_d','total_float_d','predecessor_on_time_rate','open_design_change_count','late_order_count_to_snapshot','crew_load_ratio','supplier_on_time_rate_hist','open_bim_clash_count','weather_risk_score']
FEATURES=CATEGORICAL+NUMERIC
train=data[data['split'].eq('train')].copy(); validation=data[data['split'].eq('validation')].copy(); test=data[data['split'].eq('test')].copy()

def preprocess(dense=False):
    return ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),NUMERIC),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=not dense))]),CATEGORICAL)],sparse_threshold=0.0 if dense else 0.3)

def metrics(y_true,scores,threshold):
    pred=(np.asarray(scores)>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return {'accuracy':accuracy_score(y_true,pred),'precision':precision_score(y_true,pred,zero_division=0),'recall':recall_score(y_true,pred,zero_division=0),'f1':f1_score(y_true,pred,zero_division=0),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),'predicted_positive_count':int(pred.sum())}


## 4. 逻辑回归与阈值比较

In [ ]:
logistic=Pipeline([('preprocess',preprocess()),('model',LogisticRegression(C=0.1,max_iter=3000,solver='liblinear',random_state=2026081207))])
logistic.fit(train[FEATURES],train[TARGET])
validation_scores=logistic.predict_proba(validation[FEATURES])[:,1]
current=metrics(validation[TARGET],validation_scores,THRESHOLD)
current.update({'threshold':THRESHOLD,'AP':average_precision_score(validation[TARGET],validation_scores),'ROC_AUC':roc_auc_score(validation[TARGET],validation_scores)})
display(pd.DataFrame([current]).round(3))


## 5. 学生记录与解释

先运行阈值0.50，再改为0.28。记录两次的TP、FP、FN、Precision、Recall和名单规模。

**阈值0.50：** ＿＿＿＿＿＿

**阈值0.28：** ＿＿＿＿＿＿

**变化解释：** ＿＿＿＿＿＿

## Checks

In [ ]:
assert 0<=THRESHOLD<=1
assert len(train)+len(validation)+len(test)==len(data)
print('结构检查通过；请完成两个阈值的比较。')


## Next Steps

1. 比较两个阈值的漏报和复核量；
2. 说明阈值必须在验证集确定；
3. 写出模型输出的使用边界。